# Himalayan Climbing Expeditions (1905-2019)

Database and SQL reports

## 1. Database design

We use SQLite. The data is split into 4 related tables:

- `peaks` (peak_id is the primary key)
- `agencies` (agency_id is the primary key)
- `expeditions` (expedition_id is the primary key, peak_id and agency_id are foreign keys)
- `members` (member_id is the primary key, expedition_id is a foreign key)

One peak has many expeditions, one agency has many expeditions and one expedition has many members. The table creation code is in `sql/schema.sql`, the queries are in `sql/queries.sql` and the ERD is in `reports/ERD.png`.

## 2. Loading the data into the database

The cleaned csv files are read, the tables are created from `sql/schema.sql` and the data is loaded with SQLAlchemy. Running this notebook again recreates the database.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, event

engine = create_engine("sqlite:///../database/himalayan.db")

@event.listens_for(engine, "connect")
def foreign_keys_on(dbapi_connection, connection_record):
    dbapi_connection.execute("PRAGMA foreign_keys = ON")

### Create the tables

In [2]:
schema = open("../sql/schema.sql").read()

with engine.begin() as conn:
    for statement in schema.split(";"):
        if statement.strip():
            conn.exec_driver_sql(statement)

### Read the cleaned data and fix the types for the database

In [3]:
peaks = pd.read_csv("../data/processed/peaks.csv")
agencies = pd.read_csv("../data/processed/agencies.csv")
expeditions = pd.read_csv("../data/processed/expeditions.csv")
members = pd.read_csv("../data/processed/members.csv")

expeditions[["oxygen_used", "success"]] = expeditions[["oxygen_used", "success"]].astype(int)
bool_cols = ["hired", "success", "solo", "oxygen_used", "died", "injured"]
members[bool_cols] = members[bool_cols].astype(int)

for col in ["height_metres", "first_ascent_year"]:
    peaks[col] = peaks[col].astype("Int64")
for col in ["agency_id", "duration_days", "highpoint_metres"]:
    expeditions[col] = expeditions[col].astype("Int64")
for col in ["age", "highpoint_metres"]:
    members[col] = members[col].astype("Int64")

### Load the tables (parents first, then the tables that use them)

In [4]:
peaks.to_sql("peaks", engine, if_exists="append", index=False)
agencies.to_sql("agencies", engine, if_exists="append", index=False)
expeditions.to_sql("expeditions", engine, if_exists="append", index=False)
members.to_sql("members", engine, if_exists="append", index=False)

76519

### Check the loaded tables

In [5]:
for table in ["peaks", "agencies", "expeditions", "members"]:
    n = pd.read_sql(f"select count(*) as n from {table}", engine)["n"][0]
    print(table, n, "rows")

with engine.connect() as conn:
    problems = conn.exec_driver_sql("PRAGMA foreign_key_check").fetchall()
print(len(problems), "foreign key problems")

peaks 469 rows
agencies 827 rows
expeditions 10364 rows
members 76519 rows
0 foreign key problems


## 3. SQL reports

The 10 queries are in `sql/queries.sql`. They are run here and the results are saved in `reports/SQL_Reports.md`.

In [6]:
sql_text = open("../sql/queries.sql").read()
parts = sql_text.split("-- Query ")[1:]

report = "# SQL report results\n"
for part in parts:
    title, query = part.split("\n", 1)
    query = query.strip()
    result = pd.read_sql(query, engine)
    print("Query " + title)
    display(result)
    report += "\n## Query " + title + "\n\n```sql\n" + query + "\n```\n\n" + result.to_markdown(index=False) + "\n"

open("../reports/SQL_Reports.md", "w").write(report)

Query 1: Number of expeditions and success rate in each season


,season,expeditions,successful,success_rate
0,Autumn,5064,2766,54.6
1,Spring,4875,2776,56.9
2,Winter,315,143,45.4
3,Summer,108,41,38.0


Query 2: The 10 most attempted peaks and their success rate


,peak_name,height_metres,expeditions,success_rate
0,Everest,8850,2149,61.7
1,Ama Dablam,6814,1366,67.9
2,Cho Oyu,8188,1332,63.1
3,Manaslu,8163,632,54.7
4,Lhotse,8516,429,58.3
5,Dhaulagiri I,8167,383,42.6
6,Makalu,8485,356,43.0
7,Baruntse,7152,308,39.6
8,Pumori,7138,265,47.9
9,Annapurna I,8091,243,33.7


Query 3: Expeditions in each decade


...

Query 9: Trekking agencies with at least 50 expeditions


,agency_name,expeditions,success_rate,deaths
0,Asian Trekking,853,53.7,90
1,Thamserku Trekking,707,52.5,28
2,Himalayan Guides,374,70.1,34
3,Seven Summit Treks,313,64.5,33
4,Cosmo Treks,293,52.2,25
5,Monterosa Treks,243,46.1,11
6,Cho Oyu Trekking,199,54.8,17
7,Arun Treks,148,65.5,8
8,Prestige Adventure,147,57.1,6
9,Wilderness Experience,145,53.8,17


Query 10: The 10 expeditions with the most deaths


,expedition_id,peak_name,year,member_count,total_deaths,termination_reason
0,KGUR05301,Kang Guru,2005,7,18,Accident (death or serious injury)
1,MANA72101,Manaslu,1972,12,15,Accident (death or serious injury)
2,PISA94301,Pisang,1994,10,11,Success (main peak)
3,GURJ18301,Gurja Himal,2018,5,9,Accident (death or serious injury)
4,GANG71301,Gangapurna,1971,10,8,Success (main peak)
5,EVER22101,Everest,1922,13,7,"Illness, AMS, exhaustion, or frostbite"
6,DHA169101,Dhaulagiri I,1969,10,7,Accident (death or serious injury)
7,DHA469301,Dhaulagiri IV,1969,9,6,Accident (death or serious injury)
8,EVER70101,Everest,1970,34,6,Other
9,EVER74301,Everest,1974,10,6,Accident (death or serious injury)


9902